In [9]:
import time

from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import re
from selenium.webdriver import Keys, ActionChains
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [26]:
### WISE ###

from bs4 import BeautifulSoup
import re
from selenium.webdriver import Keys, ActionChains
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


options = Options()
options.headless = True
driver = webdriver.Chrome(options=options)
driver.implicitly_wait(5)

driver.get("https://wise.com/")

# Select Source Currency (USD)
source_button = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.ID, "sourceSelectedCurrency"))
)
source_button.click()

source_input_currency_field = WebDriverWait(driver, 10).until(
    EC.visibility_of_element_located((By.ID, "sourceSelectedCurrencySearch"))
)
source_input_currency_field.send_keys("USD")
source_input_currency_field.send_keys(Keys.ENTER)

# Wait for source dropdown to close
# WebDriverWait(driver, 20).until(
#     EC.invisibility_of_element_located((By.ID, "sourceSelectedCurrencySearch"))
# )

# Select Target Currency (BDT)
target_button = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.ID, "targetSelectedCurrency"))
)
target_button.click()

target_input_currency_field = WebDriverWait(driver, 10).until(
    EC.visibility_of_element_located((By.ID, "targetSelectedCurrencySearch"))
)
target_input_currency_field.send_keys("BDT")
target_input_currency_field.send_keys(Keys.ENTER)

# Input Source Amount (1000)
source_input_field_amount = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, "input#source"))
)
actions = ActionChains(driver)
actions.click(source_input_field_amount).key_down(Keys.CONTROL).send_keys("a").key_up(Keys.CONTROL).send_keys(Keys.DELETE).perform()
source_input_field_amount.send_keys("1000")
source_input_field_amount.send_keys(Keys.ENTER)

# Wait for UI update
exchange_rate_button = WebDriverWait(driver, 10).until(
    EC.visibility_of_element_located((By.CSS_SELECTOR, "button[aria-describedby='rateLabel']"))
)
exchange_rate = exchange_rate_button.text.strip()

# Extract Transaction Fees
exchange_rate_button = WebDriverWait(driver, 10).until(
    EC.visibility_of_element_located((By.CSS_SELECTOR, "button[aria-describedby='rateLabel']"))
)
exchange_rate = exchange_rate_button.text.strip()  # e.g., "1 USD = 121.842 BDT"

time.sleep(5)

# Extract Transaction Fees with BeautifulSoup
print("Locating fees container...")
fees_container = WebDriverWait(driver, 10).until(
    EC.visibility_of_element_located((By.CSS_SELECTOR, ".Fees_container"))
)
fees_html = fees_container.get_attribute("outerHTML")

# Parse with BeautifulSoup
soup = BeautifulSoup(fees_html, "html.parser")
individual_fees = []

# Extract individual fees from <ul><li>
fee_items = soup.select("ul li.Fees_row")
for item in fee_items:
    fee_name = item.select_one("div:first-child span").text.strip()  # e.g., "Wire transfer fee"
    fee_amount = item.select_one("div:last-child span").text.strip()  # e.g., "6.11 USD"
    individual_fees.append(f"{fee_name}: {fee_amount}")

# Extract total fees
total_fees_row = soup.select_one("div.Fees_row:last-child")
percentage_fee = total_fees_row.select_one("strong:first-child span").text.strip()  # e.g., "Total included fees (1.52%)"
percentage_value = re.search(r'(\d+\.\d+%?)', percentage_fee).group(1) if re.search(r'(\d+\.\d+%?)', percentage_fee) else "Unknown"
flat_fee = total_fees_row.select_one("strong:last-child span").text.strip()  # e.g., "15.17 USD"

# Combine all fees
transaction_fees = f"Total: {flat_fee}, Percentage: {percentage_value}"

# Wait for transfer time section to stabilize
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, ".np-section.m-t-2"))
)
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, ".np-section.m-t-2 p.m-b-0 strong"))
)

# Extract Transfer Time with Selenium
transfer_time_container = driver.find_element(By.CSS_SELECTOR, ".tapestry-card-content .np-section.m-t-2 span[role='status']")
transfer_time_html = transfer_time_container.get_attribute("outerHTML")

# Parse transfer time with BeautifulSoup
soup = BeautifulSoup(transfer_time_html, "html.parser")
transfer_time = soup.select_one("p.m-b-0 strong").text.strip()  # e.g., "by Monday"
# Remove "by " prefix if present
transfer_time = transfer_time.replace("by ", "")



print(exchange_rate)
print(transaction_fees)
print(transfer_time)

driver.quit()

Locating fees container...
1 USD = 121.900 BDT
Connected bank account (ACH) fee: Connected bank account (ACH) fee, Our fee: Our fee, Dynamic charges: Dynamic charges, Total: 17.72 USD, Percentage: 1.77%
Thursday


In [25]:
import time

from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import re
from selenium.webdriver import Keys, ActionChains
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

options = Options()
options.headless = True
driver = webdriver.Chrome(options=options)
driver.implicitly_wait(5)

driver.get("https://www.xoom.com/en-us/usd/send-money/transfer?countryCode=BD")

WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )
time.sleep(2)  # Allow initial JavaScript rendering
print("Page loaded, URL:", driver.current_url)

try:
    cookie_button = WebDriverWait(driver, 5).until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, "button[data-testid='cookie-accept']"))  # Adjust selector
    )
    driver.execute_script("arguments[0].click();", cookie_button)
    print("Dismissed cookie banner")
except:
    pass


send_container = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CLASS_NAME, "_1y4lgjacz"))
    )
time.sleep(1)

amount_input_field = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, "text-input-amount-sending"))
    )
amount_input_field.click()
time.sleep(0.5)

# Log current value
# current_value = amount_input_field.get_attribute("value")
# print(f"Current input value: {current_value}")

amount_input_field.clear()
time.sleep(0.5)
cleared_value = amount_input_field.get_attribute("value")
print(f"Value after clear(): {cleared_value}")


actions = ActionChains(driver)
actions.click(amount_input_field).key_down(Keys.CONTROL).send_keys("a").key_up(Keys.CONTROL).send_keys(Keys.DELETE).perform()
time.sleep(1)

# Enter "1000" and press Enter
amount_input_field.send_keys("1000")
amount_input_field.send_keys(Keys.ENTER)

print("Successfully entered 1000 USD")

time.sleep(1)

receive_container = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "_1y4lgjacy"))
)

received_amount = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, "text-input-amount-receiving"))
    )
received_amount.click()
time.sleep(0.5)

receiving_value = received_amount.get_attribute("value")
print(f"Receiving input value: {receiving_value}")

# Locate the disbursement type selector container
disbursement_container = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CLASS_NAME, "_1y4lgjacz"))
)

# Navigate to the specific container with classes "_1y4lgjacy _1y4lgjafj _1y4lgja79 _1mcye514"
rate_container = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located(
        (By.CSS_SELECTOR, "div._1y4lgjacy._1y4lgjafj._1y4lgja79._1mcye514")
    )
)

# Find the <p> element with classes "_18ax91o1 _18ax91o0 _1mcye512" containing the rate
rate_element = rate_container.find_element(
    By.CSS_SELECTOR, "p._18ax91o1._18ax91o0._1mcye512"
)
rate_text = rate_element.text
print(f"Exchange Ratet: {rate_text}")

fee_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.ID, "xoom_fees_info"))
        )
driver.execute_script("arguments[0].click();", fee_button)
time.sleep(1)

# Extract PYUSD transaction fee from div._1y4lgjacz._1y4lgjad0 > p._18ax91o1._18ax91o0
pyusd_fee = 0.0  # Default in case element is not found
fee_containers = WebDriverWait(driver, 10).until(
    EC.presence_of_all_elements_located(
        (By.CSS_SELECTOR, "div._1y4lgjacz._1y4lgjad0")
    )
)

fees = []
for container in fee_containers:
    try:
        fee_element = container.find_element(
            By.CSS_SELECTOR, "p._18ax91o1._18ax91o0"
        )
        fee_text = fee_element.text.strip()
        try:
            fee_value = float(fee_text)
            fees.append(fee_value)
            print(f"Found fee: {fee_text} USD")
        except ValueError:
            continue  # Skip non-numeric fees
    except:
        continue  # Try next container if <p> not found

# Find and print the minimum fee
min_fee = min(fees) if fees else 0.0
print(f"Transaction Fee: {min_fee} USD")  # Try next container if <p> not found



driver.quit()


Page loaded, URL: https://www.xoom.com/en-us/usd/send-money/transfer?countryCode=BD
Value after clear(): 
Successfully entered 1000 USD
Receiving input value: 119,399.50
Exchange Ratet: 1 USD = 119.3995 BDT
Transaction Fee: 0.0 USD
